In [ ]:
# ============================================
#   Lab Assignment 5 — Gaussian HMM (Colab)
#   FINAL FULLY FIXED VERSION
# ============================================

# Install dependencies (Colab)
!pip install yfinance pandas numpy matplotlib seaborn hmmlearn scikit-learn --quiet

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler

%matplotlib inline
sns.set(style="whitegrid")

# --------------------------
# User settings
# --------------------------
TICKER = "^GSPC"          # Change to AAPL, TSLA, BTC-USD, ^NSEI, etc.
START_DATE = "2015-01-01"
END_DATE = None
N_STATES = 2
RANDOM_STATE = 42
MAX_ITER = 200
SHOW_PLOTS = True
OUT_DIR = "hmm_outputs"

os.makedirs(OUT_DIR, exist_ok=True)

# --------------------------
# FIXED download function
# --------------------------
def download_data(ticker: str, start: str, end: str = None):
    """Download stock data in a Colab-safe way."""
    print(f"Downloading {ticker} from {start} to {end or 'today'}...")

    df = yf.download(
        ticker,
        start=start,
        end=end,
        progress=False,
        auto_adjust=False
    )

    if df.empty:
        raise ValueError("Data download failed. Try another ticker.")

    print("Columns returned:", df.columns.tolist())

    # Always use Close -> adj_close (guaranteed to exist)
    if "Close" in df.columns:
        df = df[["Close"]].rename(columns={"Close": "adj_close"})
    else:
        raise KeyError("ERROR: Yahoo Finance did NOT return 'Close' column.")

    print(df.head())
    return df

# --------------------------
# Returns computation
# --------------------------
def compute_daily_returns(df):
    df = df.copy()
    df["log_ret"] = np.log(df["adj_close"]).diff()
    df = df.dropna()
    return df

# --------------------------
# Fit HMM
# --------------------------
def fit_gaussian_hmm(returns, n_states, seed, max_iter):
    model = GaussianHMM(
        n_components=n_states,
        covariance_type="diag",
        n_iter=max_iter,
        random_state=seed
    )
    model.fit(returns)
    return model

# --------------------------
# Describe states
# --------------------------
def describe_states(model):
    means = model.means_.flatten()
    variances = model.covars_.flatten()
    return means, variances

# --------------------------
# Decode HMM states
# --------------------------
def decode_states(model, data):
    hidden_states = model.predict(data)
    probs = model.predict_proba(data)
    return hidden_states, probs

# --------------------------
# Plot market regimes
# --------------------------
def plot_regimes(df, hidden_states, ticker):
    df2 = df.copy()
    df2["state"] = hidden_states

    states = sorted(df2["state"].unique())
    colors = sns.color_palette("tab10", len(states))

    fig, ax = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

    # Price series with shading
    ax[0].plot(df2.index, df2["adj_close"], lw=1)
    ax[0].set_title(f"{ticker} Price with HMM Regimes")
    ax[0].set_ylabel("Price")

    current = df2["state"].iloc[0]
    start_idx = df2.index[0]

    for idx, state in zip(df2.index[1:], df2["state"].iloc[1:]):
        if state != current:
            ax[0].axvspan(start_idx, idx, alpha=0.1, color=colors[current])
            start_idx = idx
            current = state
    ax[0].axvspan(start_idx, df2.index[-1], alpha=0.1, color=colors[current])

    # Returns scatter
    for s in states:
        mask = df2["state"] == s
        ax[1].scatter(df2.index[mask], df2["log_ret"][mask], s=10, label=f"State {s}")
    ax[1].legend()
    ax[1].set_title("Daily Returns by State")

    plt.tight_layout()
    plt.show()

# --------------------------
# Simple next-state forecast
# --------------------------
def forecast_next_state(model, last_state):
    probs = model.transmat_[last_state]
    next_state = int(np.argmax(probs))
    return next_state, probs

# --------------------------
# MAIN PIPELINE
# --------------------------
df = download_data(TICKER, START_DATE, END_DATE)
df = compute_daily_returns(df)
print("Rows after returns:", len(df))

observations = df["log_ret"].values.reshape(-1, 1)

# Scale returns
scaler = StandardScaler()
obs_scaled = scaler.fit_transform(observations)

print("Fitting HMM...")
model = fit_gaussian_hmm(obs_scaled, N_STATES, RANDOM_STATE, MAX_ITER)
print("Model trained.")

# State parameters
means, vars_ = describe_states(model)
print("\n=== Hidden State Parameters ===")
for i in range(N_STATES):
    print(f"State {i}: mean={means[i]:.5f}, var={vars_[i]:.5f}")

print("\n=== Transition Matrix ===")
print(model.transmat_)

# Decode
hidden_states, post_prob = decode_states(model, obs_scaled)
df["state"] = hidden_states

# Plot
plot_regimes(df, hidden_states, TICKER)

# Print real return stats
print("\n=== Real Return Stats by State ===")
for s in range(N_STATES):
    mask = df["state"] == s
    print(f"\nState {s}:")
    print(" Mean :", df["log_ret"][mask].mean())
    print(" Std  :", df["log_ret"][mask].std())

# Forecast next regime
last_state = int(hidden_states[-1])
next_state, next_probs = forecast_next_state(model, last_state)

print("\n=== Next Day Regime Forecast ===")
print(f"Current state: {last_state}")
print(f"Most likely next state: {next_state}")
print(f"Probabilities: {np.round(next_probs,3)}")

# =============================
# Extra Visualizations
# =============================

import seaborn as sns
import matplotlib.pyplot as plt

# 1. State Sequence Plot
def plot_state_sequence(df):
    plt.figure(figsize=(16,4))
    plt.plot(df.index, df['state'], lw=1)
    plt.title("Decoded Hidden State Sequence")
    plt.ylabel("State")
    plt.xlabel("Date")
    plt.show()


# 2. KDE Return Distributions for Each State
def plot_state_distributions(df):
    plt.figure(figsize=(10,6))
    for s in sorted(df['state'].unique()):
        sns.kdeplot(df[df['state']==s]['log_ret'], label=f"State {s}", fill=True)
    plt.title("Return Distributions by State")
    plt.xlabel("Log Return")
    plt.legend()
    plt.show()


# 3. Transition Matrix Heatmap
def plot_transition_heatmap(model):
    plt.figure(figsize=(6,5))
    sns.heatmap(model.transmat_, annot=True, fmt=".3f", cmap="Blues")
    plt.title("Transition Matrix Heatmap")
    plt.ylabel("From State")
    plt.xlabel("To State")
    plt.show()


# 4. Raw Return Scatter (Clean & Separate)
def plot_return_scatter(df):
    plt.figure(figsize=(14,5))
    sns.scatterplot(x=df.index, y=df['log_ret'], hue=df['state'], s=15, palette="tab10")
    plt.title("Daily Returns Colored by State")
    plt.xlabel("Date")
    plt.ylabel("Log Return")
    plt.legend()
    plt.show()

plot_state_sequence(df)                      # Figure 2
plot_state_distributions(df)                 # Figure 3
plot_transition_heatmap(model)               # Figure 4
plot_return_scatter(df)                      # Figure 5


# Save
df.to_csv("decoded_states_colab.csv")
print("\nSaved decoded_states_colab.csv")
